# 04 — NLP Introduction
**Goal:** Understand what NLP is and see the full pipeline in action.

Natural Language Processing (NLP) is the discipline of getting computers to read, interpret, and extract meaning from human language. Resumes are unstructured text, but they follow a very stable template — contact block, headline, skills list, experience with action-verb bullets — which makes them an unusually tractable NLP target. A well-built pipeline converts a raw resume export into structured fields (name, title, skills, employers) that an ATS can index and match against job descriptions.

This chapter runs the whole pipeline on a realistic resume in a few lines of spaCy, then demonstrates why preprocessing *order* matters. Every later chapter in this block (05–13) unpacks one stage in depth; treat this notebook as the map.

**Why it matters for resumes / ATS:** an ATS does not "read" a resume — it extracts and matches keywords and structured attributes. Each pipeline stage converts free text into a queryable record: *who* worked *where*, doing *what*, with *which skills*.

| Pipeline Stage | Input | Output | Chapter |
|---|---|---|---|
| Preprocessing | Raw text | Clean text | Ch. 06 |
| Tokenization | Clean text | Word tokens | Ch. 05 |
| POS Tagging | Tokens | Grammatical labels | Ch. 10 |
| NER | Tokens | Entity spans (PERSON, ORG, SKILL) | Ch. 12 |
| Dependency Parsing | Tagged tokens | Grammatical relationships | Ch. 11 |
| Extraction | Structured spans | ATS-ready fields | Ch. 13 |

## 1. The NLP Pipeline Overview

![NLP Pipeline Overview](../../../assets/images/nlp_pipeline_diagram_1785491141457.png)

> **Figure:** The NLP processing pipeline — from raw resume text to structured data. Each stage in this notebook maps to a step in this pipeline.

The pipeline is strictly sequential because each stage feeds the next: **preprocessing** cleans raw text, **tokenization** splits it into words, **POS tagging** labels each word, **NER** finds real-world entities (people, companies, dates), and **parsing** recovers grammatical structure before final **extraction** produces structured fields. Skipping a stage — or running it out of order — silently degrades everything downstream.

## 2. The NLP Pipeline in Action

This cell runs the entire pipeline on a realistic resume in about five lines: spaCy's single `nlp(resume)` call performs tokenization, POS tagging, NER, and dependency parsing in one pass.

**What the code does:**
1. Loads the small English model via `spacy.load("en_core_web_sm")`
2. Parses the multi-line `resume` string
3. Counts sections using regex split
4. Prints the first 8 entities with their labels
5. Counts non-space tokens

**Expected output:**
- Entities: ~13 (names, orgs, dates, skills)
- Tokens: ~56 non-space tokens
- Notice: some labels are wrong (e.g., `Python → GPE`, `TensorFlow → ORG`) — the base model guesses on spans it was never trained on

**Try it:** the mislabeled entities are the motivation for Ch. 12's custom skill extraction.

In [ ]:
import re, spacy

nlp = spacy.load("en_core_web_sm")
resume = """Srivatsa Gorti | srivatsa@email.com
Senior Data Scientist with 5+ years experience in Python, TensorFlow.
Expert in NLP and Machine Learning.

EXPERIENCE
Google, Mountain View CA — Senior Data Scientist, 2020-Present
- Developed NLP pipelines processing 10M+ documents daily
- Reduced model latency by 40% through optimization
"""

doc = nlp(resume)
print(f"Sections: {len(re.split(r'\\n{2,}', resume))}")
print(f"Entities: {len(doc.ents)}")
print(f"\n{'Entity':<25} {'Label':<10} {'Correct?'}")
print("-" * 50)
for e in doc.ents[:8]:
    print(f"{e.text:<25} {e.label_:<10} {'✓' if e.label_ in ('PERSON','ORG','DATE') else '?'}")
print(f"\nTokens: {len([t for t in doc if not t.is_space])}")

**Observation:** The base model correctly identifies `Srivatsa Gorti` as PERSON and `Google` as ORG. But it mislabels `Python` as GPE (location), `TensorFlow` as ORG, and `Machine Learning` as PERSON. These are skills, not entities — and spaCy's default model has no SKILL label. This is the core problem Ch. 12 solves with custom NER.

## 3. Text Preprocessing Order Matters

Preprocessing steps are not commutative: lowercasing before expanding contractions destroys the apostrophe that the expansion depends on. The demo sentence "I wasn't loving the team's performance... But NOW I do!" is run through two orderings so the difference is visible.

| Order | Steps | Result |
|---|---|---|
| ❌ Wrong | lowercase → strip punctuation | `wasn't` → `wasnt` (contraction lost) |
| ✓ Correct | expand contractions → lowercase → strip | `wasn't` → `was not` → `was not` |

In [ ]:
text = "I wasn't loving the team's performance... But NOW I do!"

# Wrong order: lowercase first
wrong = re.sub(r"[^\\w\\s]", "", text.lower())
print(f"Wrong (lowercase first): '{wrong}'")

# Right order: expand contractions first
right = re.sub(r"n't", " not", text)
right = re.sub(r"'s", " is", right)
right = re.sub(r"[^\\w\\s]", "", right.lower())
print(f"Right (contractions first): '{right}'")

**⚠️ Bug note:** the regex pattern `r"[^\\w\\s]"` with doubled backslashes matches literal backslash/`w`/`s` characters, not the intended whitespace/word boundary. With a correct `r'[^\w\s]'`, the right path yields readable text while the wrong path still mutilates `wasn't`. Two lessons: order matters, and always sanity-check regex escapes.

## Summary

**NLP pipeline = raw text → preprocessing → tokenization → POS → NER → parsing → extraction**

That chain is the backbone of the whole block:

1. **Preprocessing** normalizes raw text (Ch. 06)
2. **Tokenization** splits into units (Ch. 05)
3. **POS tagging** adds grammatical labels (Ch. 10)
4. **NER** finds entities (Ch. 12)
5. **Dependency parsing** recovers structure (Ch. 11)
6. **Extraction** emits structured fields (Ch. 13)

Two lessons from this chapter carry forward: stages are strictly ordered, and library defaults are only as good as the inspection you give them.